# Phase 11 — Unseen-Patient Clinical Agent (Engine 2)

Phase 11 is the conversational clinical agent. Unlike Phases 1–10, its input is a
**payload** describing a patient who is not in the cohort, rather than a `hadm_id`
whose full feature row can be looked up. That distinction is the point of the phase
and also its principal limitation, so this notebook measures it rather than assuming
it away.

## What the agent does

1. Runs the five Phase 1–5 models on the payload
2. Explains the mortality prediction with `shap.TreeExplainer`
3. Retrieves tiered evidence and composes a report under fail-closed grounding
4. Simulates counterfactuals over the supplied physiology

## What it does *not* do

**Level 5 twin retrieval is unavailable from a payload.** The Phase 7 encoder consumes
the full debiased feature set; a payload supplies labs, vitals and demographics only.
Projecting one would mean zero-filling most of the encoder input and comparing the
result against real patients, so the system refuses instead. Twin retrieval remains
available for cohort admissions via `ClinicalPromptBuilder.get_digital_twins`.

An earlier version of this notebook listed that projection as a delivered capability.
It was never functional.

## Systematic evaluation

This notebook demonstrates the agent on one patient. The evaluation across six
clinical phenotypes lives in `scripts/evaluation/run_phase11_eval.py` and
[`reports/tables/phase11_clinical_agent_evaluation.md`](../reports/tables/phase11_clinical_agent_evaluation.md).

In [1]:
import os
import sys

# Run from the project root so every relative path — models/, data/processed/,
# reports/ — resolves identically whether this is executed here or from a script.
# Pointing data_dir at '../data/processed' while leaving models_dir relative was
# enough to load no models at all.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('working directory:', os.getcwd())

import pandas as pd
import numpy as np

from src.llm.clinical_assistant import EnterpriseClinicalAgent
from src.llm.pipeline import ClinicalReportPipeline

agent = EnterpriseClinicalAgent()
pipe = ClinicalReportPipeline()
print(f'Models loaded: {sum(m is not None for m in agent.runner.lgbm_models.values())}/5')

working directory: /Users/mc/Projects/Clinical-Digital-Twin


Models loaded: 5/5


In [2]:
# A complete payload. Every field the models consume is supplied explicitly —
# omitting one silently substitutes a population-normal constant, which is
# indistinguishable from a measurement once it is in the feature vector.
unseen_patient = {
    'primary_diagnosis': 'acute kidney injury',
    'comorbidities': ['type 2 diabetes', 'chronic kidney disease stage 4'],
    'demographics': {'age': 72, 'gender': 'M'},
    'presentation_labs': {
        'creatinine_max': 4.8, 'bun_max': 88.0, 'wbc_max': 18.5,
        'bicarbonate_min': 17.0, 'sodium_min': 132.0, 'potassium_max': 5.8,
        'platelets_min': 96.0, 'hematocrit_min': 28.0, 'glucose_max': 194.0,
        'anion_gap_max': 21.0, 'chloride_max': 99.0,
    },
    'vital_signs': {'sbp_min': 90.0, 'hr_max': 118.0, 'rr_max': 26.0,
                    'spo2_min': 92.0, 'temp_max': 37.8},
    'active_medications': ['vancomycin', 'enoxaparin', 'furosemide'],
    'chief_complaint': 'Shortness of breath, decreased urine output, leg swelling',
}

# Coverage first. Everything not supplied is zero-filled, and a zero-filled feature
# is indistinguishable from a genuine zero once the design matrix is built.
print('Feature coverage from this payload')
print('-' * 46)
for task in ['mortality', 'readmission', 'icu_admission', 'hospital_los', 'deterioration']:
    print(f'  {task:16} {agent.runner.payload_feature_coverage(unseen_patient, task):6.1%}')
print()
print('The remainder are admission-derived features — diagnosis_count,')
print('procedure_count, admission-type dummies, per-analyte draw counts — which')
print('describe an admission that has not happened yet.')

Feature coverage from this payload
----------------------------------------------
  mortality         18.3%
  readmission       17.6%
  icu_admission     17.6%
  hospital_los      17.6%
  deterioration     34.1%

The remainder are admission-derived features — diagnosis_count,
procedure_count, admission-type dummies, per-analyte draw counts — which
describe an admission that has not happened yet.


In [3]:
preds = agent.tool_run_all_models(unseen_patient)

print('Multi-task predictions')
print('-' * 46)
print(f"  in-hospital mortality     {preds['p_mortality']*100:6.2f}%   {preds['risk_tier']}")
print(f"  30-day readmission        {preds['p_readmission']*100:6.2f}%")
print(f"  ICU admission             {preds['p_icu_admission']*100:6.2f}%")
print(f"  hospital LOS > 5.63 d     {preds['p_los_over_5_63d']*100:6.2f}%")
print(f"  deterioration within 6 h  {preds['p_deterioration']*100:6.2f}%")
print()
print(preds['calibration_statement'])

Multi-task predictions
----------------------------------------------
  in-hospital mortality       3.52%   Tier 3: High Risk
  30-day readmission         16.59%
  ICU admission               3.91%
  hospital LOS > 5.63 d       4.10%
  deterioration within 6 h   79.32%

The model estimates increased mortality risk based on learned patterns from the training population. This is a probabilistic estimate and not a deterministic outcome. Confidence estimated from model calibration performance.


In [4]:
# SHAP drivers. The `value` column must echo the payload — when the lab mapping was
# broken every value here was 0.0 while the numbers above still looked plausible.
drivers = agent.tool_explain_shap(unseen_patient, top_k=8)['top_shap_features']
supplied = agent.runner._convert_payload_to_series(unseen_patient)

print(f"{'feature':34}{'value':>10}{'SHAP':>10}  source")
print('-' * 74)
for d in drivers:
    src = 'payload' if d['feature'] in supplied.index else 'zero-filled'
    print(f"  {d['feature']:32}{d['value']:>10.4g}{d['shap_impact']:>+10.4f}  {src}")

feature                                value      SHAP  source
--------------------------------------------------------------------------
  diagnosis_count                          0   -1.2336  zero-filled
  lab_wbc_last_24h                      18.5   +0.5435  payload
  lab_platelets_first_24h                 96   +0.5114  payload
  lab_anion_gap_last_24h                  21   +0.5004  payload
  lab_potassium_last_24h                 5.8   +0.4792  payload
  lab_glucose_poc_missing_ratio_24h         0   +0.4359  zero-filled
  lab_bicarbonate_last_24h                17   +0.4222  payload
  lab_anion_gap_max_24h                   21   +0.3793  payload


/Users/mc/Projects/Clinical-Digital-Twin/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [5]:
# Counterfactual: normalise the renal derangements.
sim = agent.tool_simulate_counterfactual(unseen_patient, {
    'creatinine_max': 1.1, 'bun_max': 18.0,
    'bicarbonate_min': 24.0, 'potassium_max': 4.2,
})
d = sim['deltas']

print('What-if: renal panel normalised')
print('-' * 58)
print(f"  baseline        {sim['baseline_predictions']['p_mortality']*100:6.2f}%   {d['base_tier']}")
print(f"  counterfactual  {sim['counterfactual_predictions']['p_mortality']*100:6.2f}%   {d['mod_tier']}")
print(f"  delta           {d['delta_p_mortality']*100:+6.2f} pp")
print()
print('Limitation:', sim['limitation'])
print('Causal confidence:', sim['causal_confidence'], '—', sim['causal_reason'])

What-if: renal panel normalised
----------------------------------------------------------
  baseline          3.52%   Tier 3: High Risk
  counterfactual    1.20%   Tier 2: Moderate Risk
  delta            -2.33 pp

Limitation: This analysis changes selected input variables and observes model output changes. It does not simulate the biological pathway, treatment response, or causal effect of medical intervention.
Causal confidence: Not estimated — Supervised prediction models identify associations, not treatment effects.


In [6]:
# The grounded report. Fail-closed: anything not traceable to the payload, the model
# outputs or a retrieved document is withheld rather than shown.
res = pipe.generate(unseen_patient, case_id='phase11_demo', use_llm=False)

print(f"status            {res.status}")
print(f"documents         {len(res.documents)}")
print(f"generation mode   {res.generation_mode}")
print(f"grounding ok      {res.grounding.get('ok')}")
for v in (res.grounding.get('violations') or [])[:5]:
    print('   violation:', v.get('kind'), '-', v.get('detail'))

out_dir = 'reports/llm_summaries'
os.makedirs(out_dir, exist_ok=True)
path = os.path.join(out_dir, 'unseen_patient_agent_report.md')
with open(path, 'w') as f:
    f.write(res.report_markdown)
print(f'\nReport written to {path}')

status            ok
documents         6
generation mode   deterministic
grounding ok      True

Report written to reports/llm_summaries/unseen_patient_agent_report.md
